In [182]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, classification_report, confusion_matrix, accuracy_score
from sklearn.tree import DecisionTreeClassifier

Data preprocessing and merging

In [183]:
df1 = pd.read_csv("Rovaniemi Apukka_ 1.1.1844 - 15.4.2025_1a7f08c2-4d3f-4e1e-88fb-5851b6f825af.csv", encoding='utf-8')
df2 = pd.read_csv("Rovaniemi rautatieasema_ 1.1.1844 - 15.4.2025_b1a86ebc-9eea-4b9d-9014-86ec1ac75fb7.csv", encoding='utf-8')

agg_func = {'Sademäärä [mm]': 'sum', 'Lumensyvyys [cm]': 'sum', 'Ilman keskilämpötila [°C]': 'sum', 'Maanpintaminimi [°C]': 'sum', 'Ylin lämpötila [°C]': 'sum', 'Alin lämpötila [°C]': 'sum'}

df1 = df1.loc[df1['Vuosi'] >= 1950]
df1 = df1.drop(columns=['Havaintoasema'])
df1 = df1.apply(pd.to_numeric, errors='coerce')
df1 = df1.groupby(['Vuosi', 'Kuukausi', 'Päivä']).aggregate(agg_func).reset_index()

df1

,Vuosi,Kuukausi,Päivä,Sademäärä [mm],Lumensyvyys [cm],Ilman keskilämpötila [°C],Maanpintaminimi [°C],Ylin lämpötila [°C],Alin lämpötila [°C]
0,1959,1,1,0.2,32.0,-8.7,-10.4,0.0,-10.0
1,1959,1,2,-1.0,32.0,-10.3,-8.6,0.0,-15.2
2,1959,1,3,1.2,32.0,-10.9,-14.6,0.0,-14.6
3,1959,1,4,1.9,33.0,-7.8,-10.2,0.0,-9.4
4,1959,1,5,5.5,33.0,-5.5,-8.9,0.0,-7.9
...,...,...,...,...,...,...,...,...,...
24207,2025,4,11,2.1,21.0,-4.5,-9.1,-0.9,-8.5
24208,2025,4,12,0.7,21.0,-3.2,-5.8,0.8,-5.5
24209,2025,4,13,-1.0,24.0,0.8,-11.9,6.3,-5.3
24210,2025,4,14,2.2,22.0,3.8,-4.9,7.7,-2.7


In [184]:
agg_func = {'Ilman keskilämpötila [°C]': 'sum', 'Ylin lämpötila [°C]': 'sum', 'Alin lämpötila [°C]': 'sum'}

df2 = df2.drop(columns=['Havaintoasema', 'Aika [Paikallinen aika]', 'Sademäärä [mm]', 'Lumensyvyys [cm]', 'Maanpintaminimi [°C]'])
df2 = df2.apply(pd.to_numeric, errors='coerce')
df2 = df2.groupby(['Vuosi', 'Kuukausi', 'Päivä']).aggregate(agg_func).reset_index()
df2 = df2.rename(columns={'Ilman keskilämpötila [°C]': 'Ilman keskilämpötila [°C] B', 'Ylin lämpötila [°C]': 'Ylin lämpötila [°C] B', 'Alin lämpötila [°C]': 'Alin lämpötila [°C] B'})

df2

,Vuosi,Kuukausi,Päivä,Ilman keskilämpötila [°C] B,Ylin lämpötila [°C] B,Alin lämpötila [°C] B
0,1997,10,7,2.7,6.0,-0.5
1,1997,10,8,1.2,4.1,-2.6
2,1997,10,9,0.9,3.7,-1.5
3,1997,10,10,-0.5,2.6,-3.6
4,1997,10,11,1.3,2.7,-0.1
...,...,...,...,...,...,...
9976,2025,4,10,-5.1,-2.1,-9.4
9977,2025,4,11,-3.2,0.3,-6.3
9978,2025,4,12,-2.2,1.1,-5.2
9979,2025,4,13,2.3,7.2,-3.1


In [185]:
df3 = df1.merge(df2, on=['Vuosi', 'Kuukausi', 'Päivä'])
df3['Ilman keskilämpötila [°C]'] = df3[['Ilman keskilämpötila [°C]', 'Ilman keskilämpötila [°C] B']].mean(axis=1)
df3['Ylin lämpötila [°C]'] = df3[['Ylin lämpötila [°C]', 'Ylin lämpötila [°C] B']].mean(axis=1)
df3['Alin lämpötila [°C]'] = df3[['Alin lämpötila [°C]', 'Alin lämpötila [°C] B']].mean(axis=1)
df3 = df3.drop(columns=['Ilman keskilämpötila [°C] B', 'Ylin lämpötila [°C] B', 'Alin lämpötila [°C] B'])

df3

,Vuosi,Kuukausi,Päivä,Sademäärä [mm],Lumensyvyys [cm],Ilman keskilämpötila [°C],Maanpintaminimi [°C],Ylin lämpötila [°C],Alin lämpötila [°C]
0,1997,10,7,0.3,-1.0,2.80,-2.6,6.25,-0.30
1,1997,10,8,-1.0,-1.0,1.70,-3.5,4.05,-1.35
2,1997,10,9,0.1,-1.0,0.85,0.1,3.95,-1.90
3,1997,10,10,0.3,-1.0,-0.55,-8.7,2.95,-4.45
4,1997,10,11,-1.0,-1.0,1.35,-6.4,2.75,-0.40
...,...,...,...,...,...,...,...,...,...
9976,2025,4,10,-1.0,21.0,-5.60,-12.5,-2.40,-10.25
9977,2025,4,11,2.1,21.0,-3.85,-9.1,-0.30,-7.40
9978,2025,4,12,0.7,21.0,-2.70,-5.8,0.95,-5.35
9979,2025,4,13,-1.0,24.0,1.55,-11.9,6.75,-4.20


In [186]:
df = pd.concat([df1, df3]).drop_duplicates().reset_index()
df

,index,Vuosi,Kuukausi,Päivä,Sademäärä [mm],Lumensyvyys [cm],Ilman keskilämpötila [°C],Maanpintaminimi [°C],Ylin lämpötila [°C],Alin lämpötila [°C]
0,0,1959,1,1,0.2,32.0,-8.70,-10.4,0.00,-10.00
1,1,1959,1,2,-1.0,32.0,-10.30,-8.6,0.00,-15.20
2,2,1959,1,3,1.2,32.0,-10.90,-14.6,0.00,-14.60
3,3,1959,1,4,1.9,33.0,-7.80,-10.2,0.00,-9.40
4,4,1959,1,5,5.5,33.0,-5.50,-8.9,0.00,-7.90
...,...,...,...,...,...,...,...,...,...,...
34187,9976,2025,4,10,-1.0,21.0,-5.60,-12.5,-2.40,-10.25
34188,9977,2025,4,11,2.1,21.0,-3.85,-9.1,-0.30,-7.40
34189,9978,2025,4,12,0.7,21.0,-2.70,-5.8,0.95,-5.35
34190,9979,2025,4,13,-1.0,24.0,1.55,-11.9,6.75,-4.20


Adding a binary column to the dataframe depending on how snow depth changes from previous row

In [187]:
has_melted_arr = [0]

for i in range(1, len(df)):
    if df.loc[i, 'Lumensyvyys [cm]'] < df.loc[i-1, 'Lumensyvyys [cm]']:
        has_melted_arr.append(1)
    else:
        has_melted_arr.append(0)

df['hasMelted'] = has_melted_arr
df

,index,Vuosi,Kuukausi,Päivä,Sademäärä [mm],Lumensyvyys [cm],Ilman keskilämpötila [°C],Maanpintaminimi [°C],Ylin lämpötila [°C],Alin lämpötila [°C],hasMelted
0,0,1959,1,1,0.2,32.0,-8.70,-10.4,0.00,-10.00,0
1,1,1959,1,2,-1.0,32.0,-10.30,-8.6,0.00,-15.20,0
2,2,1959,1,3,1.2,32.0,-10.90,-14.6,0.00,-14.60,0
3,3,1959,1,4,1.9,33.0,-7.80,-10.2,0.00,-9.40,0
4,4,1959,1,5,5.5,33.0,-5.50,-8.9,0.00,-7.90,0
...,...,...,...,...,...,...,...,...,...,...,...
34187,9976,2025,4,10,-1.0,21.0,-5.60,-12.5,-2.40,-10.25,0
34188,9977,2025,4,11,2.1,21.0,-3.85,-9.1,-0.30,-7.40,0
34189,9978,2025,4,12,0.7,21.0,-2.70,-5.8,0.95,-5.35,0
34190,9979,2025,4,13,-1.0,24.0,1.55,-11.9,6.75,-4.20,0


In [188]:
features = ['Sademäärä [mm]', 'Lumensyvyys [cm]', 'Ilman keskilämpötila [°C]', 'Maanpintaminimi [°C]', 'Ylin lämpötila [°C]', 'Alin lämpötila [°C]']
label = 'hasMelted'

X, y = df[features].values, df[label].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)

r2 = r2_score(y_test, y_pred)

print(f"Mean squared error: {mse:.2f}")
print(f"R2 score: {r2:.2f}")

Mean squared error: 0.13
R2 score: 0.11


In [189]:
clf = DecisionTreeClassifier(max_depth=5)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

In [190]:
print(classification_report(y_test, y_pred))

print(confusion_matrix(y_test, y_pred))

print()
print(accuracy_score(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.87      0.98      0.92      8481
           1       0.76      0.31      0.44      1777

    accuracy                           0.86     10258
   macro avg       0.81      0.64      0.68     10258
weighted avg       0.85      0.86      0.84     10258

[[8303  178]
 [1225  552]]

0.8632286995515696
